# **Camada Gold**
**Aqui serão feitas operações para tornar os dados em informações analíticas pronta para consumo**

**Importando as bibliotecas**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

**Definindo o path da tabela silver**

In [0]:
path_tabela_silver = "nyc_taxi_data.silver.viagens"

**Carregando os dados da tabela silver**

In [0]:
df_silver = spark.table(path_tabela_silver)



**Verificando os dados da tabela silver**

In [0]:
df_silver.show(10)

+------------------------+-------------------+-------------------+---------------+------------------------+---------+------------------+-------------------+----------------+-----------------+-----------------------------+--------------+--------+-----------------------------+--------------+-------------+-------------------+---------------------+--------------+-------------------------+-----------+--------+-----------+------------+------------+-----------+----------+-----------------+-----------------------+---------------------+-----------------+-----------------------+-------------------+-------------------+---------------+----------------+---------------+--------------------+-------------------+------------+--------------------+----------------+
|Id_Fornecedor_Tecnologia|     Inicio_Corrida|        Fim_Corrida|Qtd_Passageiros|Distancia_corrida_milhas|Id_tarifa|Flag_armazenamento|Zona_inicio_corrida|Zona_fim_corrida|Id_tipo_pagamento|Valor_corrida_Tempo_Distancia|Taxas_Diversas|Taxa_MT

**Verificando o schema do dataframe**

In [0]:
df_silver.printSchema()

root
 |-- Id_Fornecedor_Tecnologia: integer (nullable = true)
 |-- Inicio_Corrida: timestamp_ntz (nullable = true)
 |-- Fim_Corrida: timestamp_ntz (nullable = true)
 |-- Qtd_Passageiros: long (nullable = true)
 |-- Distancia_corrida_milhas: double (nullable = true)
 |-- Id_tarifa: long (nullable = true)
 |-- Flag_armazenamento: string (nullable = true)
 |-- Zona_inicio_corrida: integer (nullable = true)
 |-- Zona_fim_corrida: integer (nullable = true)
 |-- Id_tipo_pagamento: long (nullable = true)
 |-- Valor_corrida_Tempo_Distancia: double (nullable = true)
 |-- Taxas_Diversas: double (nullable = true)
 |-- Taxa_MTA: double (nullable = true)
 |-- Valor_Gorgetas_Cartao_Credito: double (nullable = true)
 |-- Valor_Pedagios: double (nullable = true)
 |-- Taxa_Melhoria: double (nullable = true)
 |-- Valor_Total_Corrida: double (nullable = true)
 |-- Taxa_Congestionamento: double (nullable = true)
 |-- Taxa_Aeroporto: double (nullable = true)
 |-- Taxa_Congestionamento_CBD: double (nullable

**Validação rápida**

In [0]:
print(f"Total de registros na tabela Silver: {df_silver.count()}")
print(f"Total de colunas na tabela Silver: {len(df_silver.columns)}")

Total de registros na tabela Silver: 11198026
Total de colunas na tabela Silver: 42


###**Criando indicadores mensais**

In [0]:
df_gold_mensal = df_silver.groupBy(
    "Mes_Corrida",
    "Nome_Mes"
).agg(
    count("*").alias("Qtd_Viagens"),
    sum("Distancia_corrida_milhas").alias("Distancia_Total_Milhas"),
    avg("Distancia_corrida_milhas").alias("Distancia_Media_Milhas"),
    sum("Valor_Total_Corrida").alias("Receita_Total"),
    avg("Valor_Total_Corrida").alias("Valor_Medio_Corrida"),
    avg("Duracao_Viagem_Segundos").alias("Duracao_Media_Segundos"),
    sum(when(col("Status_qualidade") == "Valido", 1).otherwise(0)).alias("Qtd_Viagens_Validas"),
    sum(when(col("Status_qualidade") == "Atencao", 1).otherwise(0)).alias("Qtd_Viagens_Atencao"),
    sum(when(col("Status_qualidade") == "Invalido", 1).otherwise(0)).alias("Qtd_Viagens_Invalidas")
).orderBy("Mes_Corrida")

**Conferindo o resultado do dataframe mensal**

In [0]:
df_gold_mensal.show(10)

+-----------+--------+-----------+----------------------+----------------------+--------------------+-------------------+----------------------+-------------------+-------------------+---------------------+
|Mes_Corrida|Nome_Mes|Qtd_Viagens|Distancia_Total_Milhas|Distancia_Media_Milhas|       Receita_Total|Valor_Medio_Corrida|Duracao_Media_Segundos|Qtd_Viagens_Validas|Qtd_Viagens_Atencao|Qtd_Viagens_Invalidas|
+-----------+--------+-----------+----------------------+----------------------+--------------------+-------------------+----------------------+-------------------+-------------------+---------------------+
|          1|     Jan|    3475235|   2.034793357000437E7|      5.85512449374053| 8.900535264002399E7|  25.61131913094337|     901.0870608750199|            3324187|              87887|                63161|
|          2|     Feb|    3577542|  2.1555927230003122E7|    6.0253456786819335| 8.954767586001585E7| 25.030503026943038|     924.4705124356332|            3421781|        

**Persistindo a tabela Gold Mensal**

In [0]:
tabela_gold_mensal = "nyc_taxi_data.gold.viagens_mensal"

df_gold_mensal.write.format("delta").mode("overwrite").saveAsTable(tabela_gold_mensal)

**Checando o resultado da persistência**

In [0]:
%sql
Select * from nyc_taxi_data.gold.viagens_mensal;

Mes_Corrida,Nome_Mes,Qtd_Viagens,Distancia_Total_Milhas,Distancia_Media_Milhas,Receita_Total,Valor_Medio_Corrida,Duracao_Media_Segundos,Qtd_Viagens_Validas,Qtd_Viagens_Atencao,Qtd_Viagens_Invalidas
1,Jan,3475235,2.034793357000437E7,5.85512449374053,8.900535264002399E7,25.61131913094337,901.0870608750199,3324187,87887,63161
2,Feb,3577542,2.1555927230003122E7,6.0253456786819335,8.954767586001585E7,25.030503026943038,924.4705124356332,3421781,100489,55272
3,Mar,4145225,2.7292690060006153E7,6.584127534695018,1.0887807986003478E8,26.265903505849447,959.3522107967601,3955980,120478,68767
4,Apr,2,14.08,7.04,73.4,36.7,1283.0,2,0,0
12,Dec,22,79.81,3.627727272727273,611.9199999999998,27.81454545454545,1008.8636363636364,22,0,0


###**Criando indicadores por zonas de origem da corrida**

In [0]:
df_gold_zona_origem = df_silver.groupBy(
    "Zona_inicio_corrida",
    "Distrito_inicio",
    "Zona_inicio",
    "Zona_servico_inicio"
).agg(
    count("*").alias("Qtd_Viagens"),
    sum("Distancia_corrida_milhas").alias("Distancia_Total_Milhas"),
    avg("Distancia_corrida_milhas").alias("Distancia_Media_Milhas"),
    sum("Valor_Total_Corrida").alias("Receita_Total"),
    avg("Valor_Total_Corrida").alias("Valor_Medio_Corrida"),
    avg("Duracao_Viagem_Segundos").alias("Duracao_Media_Segundos"),
    sum(
        when(col("Status_qualidade") == "Valido", 1).otherwise(0)
    ).alias("Qtd_Viagens_Validas"),
    sum(
        when(col("Status_qualidade") == "Atencao", 1).otherwise(0)
    ).alias("Qtd_Viagens_Atencao"),
    sum(
        when(col("Status_qualidade") == "Invalido", 1).otherwise(0)
    ).alias("Qtd_Viagens_Invalidas")
).orderBy(
    col("Qtd_Viagens").desc()
)

**Conferindo o resultado do dataframe por zona de origem**

In [0]:
df_gold_zona_origem.show(20, truncate=False)

+-------------------+---------------+----------------------------+-------------------+-----------+----------------------+----------------------+--------------------+-------------------+----------------------+-------------------+-------------------+---------------------+
|Zona_inicio_corrida|Distrito_inicio|Zona_inicio                 |Zona_servico_inicio|Qtd_Viagens|Distancia_Total_Milhas|Distancia_Media_Milhas|Receita_Total       |Valor_Medio_Corrida|Duracao_Media_Segundos|Qtd_Viagens_Validas|Qtd_Viagens_Atencao|Qtd_Viagens_Invalidas|
+-------------------+---------------+----------------------------+-------------------+-----------+----------------------+----------------------+--------------------+-------------------+----------------------+-------------------+-------------------+---------------------+
|161                |Manhattan      |Midtown Center              |Yellow Zone        |513560     |1912427.0800000094    |3.7238629955604203    |1.2156476370000536E7|23.670995346211807 |86

**Checando o schema do dataframe por zona de início da corrida**

In [0]:
df_gold_zona_origem.printSchema()

root
 |-- Zona_inicio_corrida: integer (nullable = true)
 |-- Distrito_inicio: string (nullable = true)
 |-- Zona_inicio: string (nullable = true)
 |-- Zona_servico_inicio: string (nullable = true)
 |-- Qtd_Viagens: long (nullable = false)
 |-- Distancia_Total_Milhas: double (nullable = true)
 |-- Distancia_Media_Milhas: double (nullable = true)
 |-- Receita_Total: double (nullable = true)
 |-- Valor_Medio_Corrida: double (nullable = true)
 |-- Duracao_Media_Segundos: double (nullable = true)
 |-- Qtd_Viagens_Validas: long (nullable = true)
 |-- Qtd_Viagens_Atencao: long (nullable = true)
 |-- Qtd_Viagens_Invalidas: long (nullable = true)



**Validando os dados**

In [0]:
print("Quantidade de zonas:", df_gold_zona_origem.count())

df_gold_zona_origem.select(
    sum("Qtd_Viagens").alias("Total_Viagens")
).show()

Quantidade de zonas: 261
+-------------+
|Total_Viagens|
+-------------+
|     11198026|
+-------------+



**Persistindo a tabela Gold por zona de início**

In [0]:
tabela_gold_zona_origem = "nyc_taxi_data.gold.viagens_zona_origem"

df_gold_zona_origem.write.format("delta").mode("overwrite").saveAsTable(tabela_gold_zona_origem)

**Checando o resultado da tabela**

In [0]:
%sql
SELECT * from nyc_taxi_data.gold.viagens_zona_origem;

Zona_inicio_corrida,Distrito_inicio,Zona_inicio,Zona_servico_inicio,Qtd_Viagens,Distancia_Total_Milhas,Distancia_Media_Milhas,Receita_Total,Valor_Medio_Corrida,Duracao_Media_Segundos,Qtd_Viagens_Validas,Qtd_Viagens_Atencao,Qtd_Viagens_Invalidas
161,Manhattan,Midtown Center,Yellow Zone,513560,1912427.0800000094,3.7238629955604203,1.2156476370000536E7,23.670995346211807,864.0029869927564,494479,10671,8410
237,Manhattan,Upper East Side South,Yellow Zone,491957,1252967.3700000069,2.5469042416308882,9677396.639999498,19.67122459889685,695.2321869594293,477204,8164,6589
236,Manhattan,Upper East Side North,Yellow Zone,456758,975113.8299999993,2.1348587873666127,9159863.919999301,20.05408535810933,717.6430473029482,443161,8959,4638
132,Queens,JFK Airport,Airports,433506,6676173.910000002,15.400418702393974,3.141359754000316E7,72.46404326584444,2353.932575327677,398428,8380,26698
230,Manhattan,Times Sq/Theatre District,Yellow Zone,377216,1324792.9600000046,3.512027485578567,9669465.309999967,25.633762380174666,925.5440039658976,361221,6741,9254
186,Manhattan,Penn Station/Madison Sq West,Yellow Zone,365697,1032411.2100000117,2.823132839481898,8460041.650000153,23.134019830625224,901.4940538205126,352037,5709,7951
162,Manhattan,Midtown East,Yellow Zone,360560,988116.6200000013,2.740505380519196,8317343.200000076,23.067847792323263,819.589696583093,347625,7179,5756
142,Manhattan,Lincoln Square East,Yellow Zone,332387,1062230.530000007,3.195764365032348,7009075.079999972,21.087091492747827,750.7266138567393,322088,5689,4610
234,Manhattan,Union Sq,Yellow Zone,312841,1078262.2500000088,3.446678184764813,6662268.879999906,21.296022196578793,780.404547357923,301387,7616,3838
170,Manhattan,Murray Hill,Yellow Zone,303493,1391886.859999997,4.586223932677186,6720904.489999911,22.145171354857972,805.1656644469559,290032,8514,4947


###**Criando a tabela com agração por zona de destino**

In [0]:
df_gold_zona_destino = df_silver.groupBy(
    "Zona_fim_corrida",
    "Distrito_fim",
    "Zona_fim",
    "Zona_servico_fim"
).agg(
    count("*").alias("Qtd_Viagens"),
    sum("Distancia_corrida_milhas").alias("Distancia_Total_Milhas"),
    avg("Distancia_corrida_milhas").alias("Distancia_Media_Milhas"),
    sum("Valor_Total_Corrida").alias("Receita_Total"),
    avg("Valor_Total_Corrida").alias("Valor_Medio_Corrida"),
    avg("Duracao_Viagem_Segundos").alias("Duracao_Media_Segundos"),
    sum(
        when(col("Status_qualidade") == "Valido", 1).otherwise(0)
    ).alias("Qtd_Viagens_Validas"),
    sum(
        when(col("Status_qualidade") == "Atencao", 1).otherwise(0)
    ).alias("Qtd_Viagens_Atencao"),
    sum(
        when(col("Status_qualidade") == "Invalido", 1).otherwise(0)
    ).alias("Qtd_Viagens_Invalidas")
).orderBy(
    col("Qtd_Viagens").desc()
)

**Checando o resultado**

In [0]:
df_gold_zona_destino.show(20, truncate=False)

+----------------+------------+-----------------------------+----------------+-----------+----------------------+----------------------+------------------+-------------------+----------------------+-------------------+-------------------+---------------------+
|Zona_fim_corrida|Distrito_fim|Zona_fim                     |Zona_servico_fim|Qtd_Viagens|Distancia_Total_Milhas|Distancia_Media_Milhas|Receita_Total     |Valor_Medio_Corrida|Duracao_Media_Segundos|Qtd_Viagens_Validas|Qtd_Viagens_Atencao|Qtd_Viagens_Invalidas|
+----------------+------------+-----------------------------+----------------+-----------+----------------------+----------------------+------------------+-------------------+----------------------+-------------------+-------------------+---------------------+
|236             |Manhattan   |Upper East Side North        |Yellow Zone     |476637     |1292496.8000000059    |2.711700518423886     |9970807.619998917 |20.91908018051246  |719.43287659162       |463555             

In [0]:
print("Quantidade de zonas:", df_gold_zona_destino.count())

Quantidade de zonas: 260


In [0]:
df_gold_zona_destino.select(
    sum("Qtd_Viagens").alias("Total_Viagens")
).show()

+-------------+
|Total_Viagens|
+-------------+
|     11198026|
+-------------+



**Persistindo a tabela de zonas destino**


In [0]:
tabela_gold_zona_destino = "nyc_taxi_data.gold.viagens_zona_destino"

df_gold_zona_destino.write.format("delta").mode("overwrite").saveAsTable(tabela_gold_zona_destino)

**Conferindo o resultado**

In [0]:
%sql
SELECT * FROM nyc_taxi_data.gold.viagens_zona_destino;

Zona_fim_corrida,Distrito_fim,Zona_fim,Zona_servico_fim,Qtd_Viagens,Distancia_Total_Milhas,Distancia_Media_Milhas,Receita_Total,Valor_Medio_Corrida,Duracao_Media_Segundos,Qtd_Viagens_Validas,Qtd_Viagens_Atencao,Qtd_Viagens_Invalidas
236,Manhattan,Upper East Side North,Yellow Zone,476637,1292496.8000000059,2.711700518423886,9970807.619998917,20.91908018051246,719.43287659162,463555,7760,5322
237,Manhattan,Upper East Side South,Yellow Zone,447398,1483341.5300000147,3.315485384378148,8872650.709999707,19.83167271646209,692.1287980724098,433154,8017,6227
161,Manhattan,Midtown Center,Yellow Zone,399919,1773564.2200000046,4.434808598741257,9093137.690000083,22.73744855833327,853.0403481705046,384656,8295,6968
230,Manhattan,Times Sq/Theatre District,Yellow Zone,337966,1397608.9800000123,4.135353792985129,9089305.209999662,26.894140860322228,1036.4514004367304,319170,10357,8439
170,Manhattan,Murray Hill,Yellow Zone,319626,1606705.0100000096,5.026828261780986,7071758.719999775,22.12510471613628,777.5572168722194,308007,6709,4910
142,Manhattan,Lincoln Square East,Yellow Zone,298235,1057520.170000003,3.5459291163009135,6457990.36000001,21.654032424095124,776.0073432025081,287110,6896,4229
239,Manhattan,Upper West Side South,Yellow Zone,297133,1355159.44000001,4.560784026008589,6728498.169999961,22.64473542151145,788.9793493149532,287568,6090,3475
162,Manhattan,Midtown East,Yellow Zone,294507,1523727.1600000076,5.1738232367991515,6645471.379999851,22.564731500439212,796.6957967043228,283768,5908,4831
68,Manhattan,East Chelsea,Yellow Zone,290755,1546300.3700000192,5.318224518924934,6634671.809999846,22.818771164725785,832.0973259273271,278475,7558,4722
141,Manhattan,Lenox Hill West,Yellow Zone,284415,1633506.4700000114,5.743390714273197,6077319.900000055,21.36778967354062,732.3401332559815,275378,5380,3657


**Ranking mensal de distritos com Window Function**

Será utilizada uma Window Function para classificar os distritos de origem das corridas de acordo com a quantidade de viagens em cada mês.

A janela será particionada por mês e ordenada pela quantidade de viagens em ordem decrescente.

In [0]:
df_distrito_mes = (
    df_silver
    .groupBy(
        "Mes_Corrida",
        "Nome_Mes",
        "Distrito_inicio"
    )
    .agg(
        count("*").alias("Qtd_Viagens"),
        sum("Valor_Total_Corrida").alias("Receita_Total"),
        avg("Distancia_corrida_milhas").alias("Distancia_Media_Milhas")
    )
)

In [0]:
janela_ranking = (
    Window
    .partitionBy("Mes_Corrida")
    .orderBy(col("Qtd_Viagens").desc())
)

In [0]:
df_gold_ranking_distritos = (
    df_distrito_mes
    .withColumn(
        "Ranking_Distrito",
        dense_rank().over(janela_ranking)
    )
    .orderBy(
        "Mes_Corrida",
        "Ranking_Distrito"
    )
)

In [0]:
df_gold_ranking_distritos.show(30, truncate=False)

+-----------+--------+---------------+-----------+--------------------+----------------------+----------------+
|Mes_Corrida|Nome_Mes|Distrito_inicio|Qtd_Viagens|Receita_Total       |Distancia_Media_Milhas|Ranking_Distrito|
+-----------+--------+---------------+-----------+--------------------+----------------------+----------------+
|1          |Jan     |Manhattan      |3089285    |6.709038546001529E7 |4.44391060714603      |1               |
|1          |Jan     |Queens         |294985     |1.9267557509998076E7|13.361014560061776    |2               |
|1          |Jan     |Brooklyn       |66070      |1791676.6699999792  |24.807041925231157    |3               |
|1          |Jan     |Bronx          |14741      |462018.13000000257  |65.91340478936304     |4               |
|1          |Jan     |Unknown        |8141       |225274.0600000007   |3.202056258444915     |5               |
|1          |Jan     |N/A            |1380       |123597.05999999972  |28.235833333333336    |6         

**Persistindo a tabela**

In [0]:
tabela_gold_ranking_distritos = "nyc_taxi_data.gold.ranking_mensal_distritos"

df_gold_ranking_distritos.write.format("delta").mode("overwrite").saveAsTable(tabela_gold_ranking_distritos)

**Checando o resultado**

In [0]:
%sql
SELECT *
FROM nyc_taxi_data.gold.ranking_mensal_distritos
ORDER BY Mes_Corrida, Ranking_Distrito;

Mes_Corrida,Nome_Mes,Distrito_inicio,Qtd_Viagens,Receita_Total,Distancia_Media_Milhas,Ranking_Distrito
1,Jan,Manhattan,3089285,6.709038546001529E7,4.44391060714603,1
1,Jan,Queens,294985,1.9267557509998076E7,13.361014560061776,2
1,Jan,Brooklyn,66070,1791676.6699999792,24.807041925231157,3
1,Jan,Bronx,14741,462018.13000000257,65.91340478936304,4
1,Jan,Unknown,8141,225274.0600000007,3.202056258444915,5
1,Jan,N/A,1380,123597.05999999972,28.235833333333336,6
1,Jan,EWR,377,35565.38000000001,0.8962864721485414,7
1,Jan,Staten Island,256,9278.369999999997,8.305703124999999,8
2,Feb,Manhattan,3158873,6.892741385000508E7,4.198094909164033,1
2,Feb,Queens,291134,1.7073750799995154E7,12.624687360459154,2


###Pivot - Quantidade de viagens por distrito e mês

Será utilizada uma operação de pivotação para transformar os meses em colunas e facilitar a comparação da quantidade de viagens entre os distritos ao longo do período analisado.

In [0]:
meses = ["Dec", "Jan", "Feb", "Mar", "Apr"]

df_gold_pivot_distritos = (
    df_silver
    .groupBy("Distrito_inicio")
    .pivot("Nome_Mes", meses)
    .agg(
        count("Inicio_Corrida")
    )
    .fillna(0)
)

**Checando o resultado**

In [0]:
df_gold_pivot_distritos.show(truncate=False)

+---------------+---+-------+-------+-------+---+
|Distrito_inicio|Dec|Jan    |Feb    |Mar    |Apr|
+---------------+---+-------+-------+-------+---+
|Unknown        |0  |8141   |7470   |7687   |0  |
|Queens         |5  |294985 |291134 |378934 |1  |
|EWR            |0  |377    |314    |365    |0  |
|Manhattan      |17 |3089285|3158873|3578937|1  |
|N/A            |0  |1380   |1165   |1548   |0  |
|Staten Island  |0  |256    |305    |429    |0  |
|Bronx          |0  |14741  |21466  |34199  |0  |
|Brooklyn       |0  |66070  |96815  |143126 |0  |
+---------------+---+-------+-------+-------+---+



**Validando o total**

In [0]:
df_gold_pivot_distritos.select(
    (
        col("Dec") +
        col("Jan") +
        col("Feb") +
        col("Mar") +
        col("Apr")
    ).alias("Total_Linha")
).agg(
    sum("Total_Linha").alias("Total_Viagens")
).show()

+-------------+
|Total_Viagens|
+-------------+
|     11198026|
+-------------+



**Persistindo a Pivot**

In [0]:
tabela_gold_pivot_distritos = "nyc_taxi_data.gold.viagens_pivot_distrito_mes"

df_gold_pivot_distritos.write.format("delta").mode("overwrite").saveAsTable(tabela_gold_pivot_distritos)

**Checando o resultado**


In [0]:
%sql
SELECT *
FROM nyc_taxi_data.gold.viagens_pivot_distrito_mes;

Distrito_inicio,Dec,Jan,Feb,Mar,Apr
Manhattan,17,3089285,3158873,3578937,1
Queens,5,294985,291134,378934,1
Unknown,0,8141,7470,7687,0
Staten Island,0,256,305,429,0
Bronx,0,14741,21466,34199,0
N/A,0,1380,1165,1548,0
Brooklyn,0,66070,96815,143126,0
EWR,0,377,314,365,0


**Resultado da pivotação**

A operação de pivotação reorganizou a quantidade de viagens por distrito de origem, transformando os meses em colunas. A validação da soma das colunas resultou em 11.198.026 viagens, correspondente ao total de registros da camada Silver.

### UDF - Classificação das viagens por distância

Será utilizada uma User Defined Function (UDF) para classificar as viagens em categorias de distância. A função recebe a distância da corrida em milhas e retorna sua respectiva classificação.

In [0]:
def classificar_distancia(distancia):
    if distancia is None:
        return "Nao Informado"
    elif distancia <= 2:
        return "Curta"
    elif distancia <= 5:
        return "Media"
    elif distancia <= 10:
        return "Longa"
    else:
        return "Muito Longa"

udf_classificar_distancia = udf(
    classificar_distancia,
    StringType()
)

In [0]:
df_viagens_classificadas = (
    df_silver
    .withColumn(
        "Categoria_Distancia",
        udf_classificar_distancia(col("Distancia_corrida_milhas"))
    )
)

In [0]:
df_viagens_classificadas.select(
    "Distancia_corrida_milhas",
    "Categoria_Distancia"
).show(20, truncate=False)

+------------------------+-------------------+
|Distancia_corrida_milhas|Categoria_Distancia|
+------------------------+-------------------+
|0.9                     |Curta              |
|0.6                     |Curta              |
|1.94                    |Curta              |
|0.95                    |Curta              |
|1.5                     |Curta              |
|2.0                     |Curta              |
|3.27                    |Media              |
|0.95                    |Curta              |
|2.09                    |Media              |
|1.43                    |Curta              |
|0.89                    |Curta              |
|0.72                    |Curta              |
|18.6                    |Muito Longa        |
|23.28                   |Muito Longa        |
|2.73                    |Media              |
|1.29                    |Curta              |
|1.99                    |Curta              |
|0.9                     |Curta              |
|2.4         

In [0]:
df_gold_categoria_distancia = (
    df_viagens_classificadas
    .groupBy("Categoria_Distancia")
    .agg(
        count("Inicio_Corrida").alias("Qtd_Viagens"),
        avg("Distancia_corrida_milhas").alias("Distancia_Media_Milhas"),
        avg("Valor_Total_Corrida").alias("Valor_Medio_Corrida"),
        sum("Valor_Total_Corrida").alias("Receita_Total")
    )
    .orderBy(col("Qtd_Viagens").desc())
)

In [0]:
df_gold_categoria_distancia.show(truncate=False)


+-------------------+-----------+----------------------+-------------------+--------------------+
|Categoria_Distancia|Qtd_Viagens|Distancia_Media_Milhas|Valor_Medio_Corrida|Receita_Total       |
+-------------------+-----------+----------------------+-------------------+--------------------+
|Curta              |6413515    |1.0869611437711841    |17.002243554394063 |1.0904414406975964E8|
|Media              |3033957    |3.032616553234434     |25.52147030100785  |7.743104347003487E7 |
|Longa              |1008277    |7.192102904261477     |42.329113398399    |4.267947146999755E7 |
|Muito Longa        |742277     |61.66558170332615     |78.51130328703469  |5.827713466999025E7 |
+-------------------+-----------+----------------------+-------------------+--------------------+



**Persistindo o df criado a partir da UDF**

In [0]:
tabela_gold_categoria_distancia = "nyc_taxi_data.gold.viagens_categoria_distancia"

df_gold_categoria_distancia.write.format("delta").mode("overwrite").saveAsTable(tabela_gold_categoria_distancia)

**Checando o resultado**

In [0]:
%sql
SELECT *
FROM nyc_taxi_data.gold.viagens_categoria_distancia
ORDER BY Qtd_Viagens DESC;

Categoria_Distancia,Qtd_Viagens,Distancia_Media_Milhas,Valor_Medio_Corrida,Receita_Total
Curta,6413515,1.0869611437711841,17.002243554394063,1.0904414406975964E8
Media,3033957,3.032616553234434,25.52147030100785,7.743104347003487E7
Longa,1008277,7.192102904261477,42.329113398399,4.267947146999755E7
Muito Longa,742277,61.66558170332615,78.51130328703469,5.827713466999025E7


**Resultado da classificação por distância**

A UDF classificou as viagens em quatro categorias de acordo com a distância percorrida: Curta, Media, Longa e Muito Longa.

Após a aplicação da função, os dados foram agregados para obtenção da quantidade de viagens, distância média, valor médio e receita total por categoria.

A soma das categorias resultou em 11.198.026 viagens, correspondente ao total de registros da camada Silver.